In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from sqlalchemy import create_engine

# Update with YOUR MySQL credentials
USER     = "root"
PASSWORD = "Your Password"
HOST     = "localhost"
PORT     = "3306"
DATABASE = "your database name"   # your DB name

engine = create_engine(
    f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"
)
print("Connected to MySQL ✓")

Connected to MySQL ✓


In [14]:
fake = Faker('en_IN')
np.random.seed(42)
N = 50_000

CATEGORIES = ['Retail', 'Textile', 'FMCG', 'Pharma', 'IT Services','Construction','Food Processing']
STATES = ['Maharashtra','Gujarat','Tamil Nadu','Rajasthan','West Bengal','Karnataka']

# --- Table 1: msme_profiles ---
msme_ids = [f'MSME{i:06d}' for i in range(N)]
df_profiles = pd.DataFrame({
    'msme_id': msme_ids,
    'business_name':[fake.company() for _ in range(N)],
    'category': random.choices(CATEGORIES, k=N),
    'business_age_yrs': np.round(np.random.gamma(3,2.5,N),1),
    'state': random.choices(STATES, k=N),
    'gst_registered': np.random.choice([1,0],p=[0.72,0.28],size=N),
    'annual_turnover': np.random.lognormal(13.5,1.2,N).round(2),
    'employee_count': np.random.randint(1,250,N)
})

# --- Table 2: loan_records ---
DEFAULT_RATE = {
    'Retail':0.12,'Textile':0.21,'FMCG':0.09,
    'Pharma':0.06,'IT Services':0.04,
    'Construction':0.18,'Food Processing':0.14
}
cats = df_profiles['category'].tolist()
delinq = [1 if random.random() < DEFAULT_RATE[c] else 0 for c in cats]
disbursement = pd.date_range('2023-01-01','2025-12-31',periods=N)

df_loans = pd.DataFrame({
    'loan_id': [f'LN{i:07d}' for i in range(N)],
    'msme_id': msme_ids,
    'loan_amount': np.random.lognormal(12,1.1,N).round(2),
    'disbursement_dt': disbursement.strftime('%Y-%m-%d'),
    'tenure_days': np.random.choice([90,180,270,365],size=N),
    'interest_rate': np.random.uniform(11,24,N).round(2),
    'loan_status': ['NPA' if d else random.choice(['Active','Closed'])
                    for d in delinq],
    'delinquency_flag': delinq
})

# --- Table 3: txn_logs (monthly records for each MSME) ---
txn_rows = []
for mid in msme_ids[:10000]:  # 10K MSMEs × 12 months
    base_vol = np.random.lognormal(11, 1)
    for m in pd.date_range('2024-01-01', periods=12, freq='MS'):
        vol = max(0, base_vol * np.random.normal(1, 0.25))
        txn_rows.append({
            'msme_id': mid,
            'txn_month': m.strftime('%Y-%m-%d'),
            'monthly_txn_vol': round(vol, 2),
            'txn_count': int(np.random.poisson(45)),
            'avg_txn_value': round(vol / max(1, np.random.poisson(45)), 2)
        })
df_txn = pd.DataFrame(txn_rows)

# --- Table 4: repayment_history ---
repay_rows = []
for _, row in df_loans.iterrows():
    due = pd.Timestamp(row['disbursement_dt']) + pd.Timedelta(days=30)
    overdue = int(np.random.exponential(12)) if row['delinquency_flag'] else 0
    paid_dt = due + pd.Timedelta(days=overdue)
    repay_rows.append({
        'loan_id': row['loan_id'],
        'due_date': due.strftime('%Y-%m-%d'),
        'paid_date': paid_dt.strftime('%Y-%m-%d') if overdue < 90 else None,
        'amount_due': round(row['loan_amount'] * 0.1, 2),
        'amount_paid': round(row['loan_amount'] * 0.1 * random.uniform(0.5,1.0),2),
        'days_overdue': overdue if overdue > 0 else None
    })
df_repay = pd.DataFrame(repay_rows)

# --- Push all tables to MySQL ---
print("Inserting msme_profiles...")
df_profiles.to_sql('msme_profiles', engine, if_exists='append', index=False, chunksize=1000)

print("Inserting loan_records...")
df_loans.to_sql('loan_records', engine, if_exists='append', index=False, chunksize=1000)

print("Inserting txn_logs...")
df_txn.to_sql('txn_logs', engine, if_exists='append', index=False, chunksize=1000)

print("Inserting repayment_history...")
df_repay.to_sql('repayment_history', engine, if_exists='append', index=False, chunksize=1000)

print("All data loaded ✓")

Inserting msme_profiles...
Inserting loan_records...
Inserting txn_logs...
Inserting repayment_history...
All data loaded ✓


In [15]:
# Pull the full analytical master view
df = pd.read_sql("SELECT * FROM v_credit_master", engine)

# Quick audit — run these 4 lines immediately
print(f"Rows : {len(df):,}")               # expect ~50,000
print(f"Cols : {df.shape[1]}")              # expect ~22 columns
print(df.isnull().sum()[df.isnull().sum() > 0])  # flag NULLs
print(df['delinquency_flag'].value_counts())     # check class balance

Rows : 50,000
Cols : 24
avg_monthly_vol    40000
txn_volatility     40000
total_txns         40000
dtype: int64
delinquency_flag
0    44097
1     5903
Name: count, dtype: int64


In [16]:
# Or run a specific EDA query directly in Python
df_sector = pd.read_sql("""
    SELECT category,
           ROUND(AVG(delinquency_flag)*100, 2) AS default_rate_pct,
           COUNT(*) AS total
    FROM v_credit_master
    GROUP BY category
    ORDER BY default_rate_pct DESC
""", engine)
print(df_sector)

          category  default_rate_pct  total
0          Textile             20.42   7239
1     Construction             18.34   7226
2  Food Processing             13.38   7084
3           Retail             11.89   7118
4             FMCG              8.54   7076
5           Pharma              5.98   7142
6      IT Services              3.87   7115
